# Laboratorio 6 – KNN  
## Pasaporte: incisos 1 al 5

**Objetivo de este notebook.**  
En este avance se construyen los modelos de **Naive Bayes** para:
1. **predicción del precio** de las viviendas, y  
2. **clasificación** de la categoría de precio (**Económica, Intermedia, Cara**),  

manteniendo exactamente la misma lógica de limpieza, la misma variable categórica y los mismos conjuntos de **entrenamiento/prueba** utilizados en el laboratorio anterior, para que las comparaciones sean válidas.


## 0. Librerias y utilidades


In [1]:

try:
    import pyreadr
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyreadr", "-q"])
    import pyreadr

import warnings
warnings.filterwarnings("ignore")

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def parse_money(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().replace("$","").replace(",","")
    try:
        return float(s)
    except:
        return np.nan

def parse_pct(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().replace("%","")
    if s == "" or s.lower() == "nan":
        return np.nan
    try:
        return float(s)
    except:
        return np.nan

def parse_numeric(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().replace(",","")
    if s == "" or s.lower() == "nan":
        return np.nan
    try:
        return float(s)
    except:
        return np.nan

def count_list_like(series):
    s = series.fillna("").astype(str)
    quoted = s.str.count(r'"')
    counts = (quoted // 2).astype(float)
    fallback = s.str.count(",") + 1
    counts = counts.where(s.str.len() > 0, np.nan)
    counts = counts.where(counts > 0, fallback.astype(float))
    return counts

def make_pre(num, cat, dense=False, scale_num=False):
    num_steps = [("imp", SimpleImputer(strategy="median"))]
    if scale_num:
        num_steps.append(("scaler", StandardScaler()))
    cat_ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=not dense)
    return ColumnTransformer([
        ("num", Pipeline(num_steps), num),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("ohe", cat_ohe)
        ]), cat)
    ], sparse_threshold=0.0 if dense else 1.0)

def prepare_dense_nb(Xtrain, Xtest, num_cols, cat_cols, scale=True):
    imp_num = SimpleImputer(strategy="median")
    Xtr_num = pd.DataFrame(imp_num.fit_transform(Xtrain[num_cols]), columns=num_cols, index=Xtrain.index)
    Xte_num = pd.DataFrame(imp_num.transform(Xtest[num_cols]), columns=num_cols, index=Xtest.index)

    if scale:
        scaler = StandardScaler()
        Xtr_num = pd.DataFrame(scaler.fit_transform(Xtr_num), columns=num_cols, index=Xtrain.index)
        Xte_num = pd.DataFrame(scaler.transform(Xte_num), columns=num_cols, index=Xtest.index)

    imp_cat = SimpleImputer(strategy="most_frequent")
    Xtr_cat = pd.DataFrame(imp_cat.fit_transform(Xtrain[cat_cols]), columns=cat_cols, index=Xtrain.index)
    Xte_cat = pd.DataFrame(imp_cat.transform(Xtest[cat_cols]), columns=cat_cols, index=Xtest.index)

    Xtr_cat = pd.get_dummies(Xtr_cat.astype(str), drop_first=False)
    Xte_cat = pd.get_dummies(Xte_cat.astype(str), drop_first=False)
    Xte_cat = Xte_cat.reindex(columns=Xtr_cat.columns, fill_value=0)

    Xtr = pd.concat([Xtr_num, Xtr_cat], axis=1).astype(float).values
    Xte = pd.concat([Xte_num, Xte_cat], axis=1).astype(float).values
    return Xtr, Xte

def nb_regression_binned(Xtr, ytr, Xte, n_bins=6, var_smoothing=1e-3):
    ytr = pd.Series(ytr).reset_index(drop=True)
    codes, bins = pd.qcut(ytr, q=n_bins, labels=False, retbins=True, duplicates="drop")
    codes = pd.Series(codes).astype(int)

    model = GaussianNB(var_smoothing=var_smoothing)
    model.fit(Xtr, codes)

    band_summary = pd.DataFrame({
        "banda": sorted(codes.unique())
    })
    band_summary["mediana_precio"] = band_summary["banda"].map(ytr.groupby(codes).median())
    band_summary["media_precio"] = band_summary["banda"].map(ytr.groupby(codes).mean())
    band_summary["n"] = band_summary["banda"].map(codes.value_counts().sort_index())

    pred_codes = model.predict(Xte)
    pred = band_summary.set_index("banda").loc[pred_codes, "mediana_precio"].values
    return pred, pred_codes, model, band_summary, bins



## 1. Carga y preprocesamiento del conjunto de datos

En esta sección se **reconstruye exactamente** la limpieza usada en el laboratorio anterior para asegurar comparabilidad:
- se convierte el precio a formato numérico,
- se generan variables derivadas útiles,
- se eliminan filas sin precio,
- y se recorta el target a precios de hasta **USD 5,000 por noche**, tal como ya se justificó en el análisis exploratorio del laboratorio previo.


In [ ]:

rdata_path = "/mnt/data/listings.RData"
result = pyreadr.read_r(rdata_path)
df_raw = list(result.values())[0].copy()

print("Filas originales:", df_raw.shape[0])
print("Columnas originales:", df_raw.shape[1])

df = df_raw.copy()

for c in ["last_scraped","host_since","calendar_last_scraped","first_review","last_review"]:
    df[c] = pd.to_datetime(df[c], errors="coerce")

df["price_num"] = df["price"].map(parse_money)
df["host_response_rate_num"] = df["host_response_rate"].map(parse_pct)
df["host_acceptance_rate_num"] = df["host_acceptance_rate"].map(parse_pct)

for c in ["beds","bedrooms","host_listings_count","host_total_listings_count",
          "minimum_minimum_nights","maximum_minimum_nights","minimum_maximum_nights",
          "maximum_maximum_nights","estimated_revenue_l365d"]:
    if c in df.columns:
        df[c] = df[c].map(parse_numeric)

df["bathrooms_num"] = df["bathrooms"].where(
    df["bathrooms"].notna(),
    df["bathrooms_text"].fillna("").astype(str).str.extract(r"(\d+(\.\d+)?)")[0].astype(float)
)

df["amenities_count"] = count_list_like(df["amenities"])
df["host_verifications_count"] = count_list_like(df["host_verifications"])
df["description_length"] = df["description"].fillna("").astype(str).str.len()
df["host_about_length"] = df["host_about"].fillna("").astype(str).str.len()
df["neighborhood_overview_length"] = df["neighborhood_overview"].fillna("").astype(str).str.len()
df["name_length"] = df["name"].fillna("").astype(str).str.len()

ref_date = df["last_scraped"].max()
df["host_tenure_days"] = (ref_date - df["host_since"]).dt.days
df["days_since_first_review"] = (ref_date - df["first_review"]).dt.days
df["days_since_last_review"] = (ref_date - df["last_review"]).dt.days

for c in ["host_is_superhost","host_has_profile_pic","host_identity_verified","has_availability","instant_bookable"]:
    if c in df.columns:
        df[c] = df[c].replace({"t":"Sí","f":"No","":"Desconocido"}).fillna("Desconocido")

df_clean = df[df["price_num"].notna()].copy()
df_clean = df_clean[df_clean["price_num"] <= 5000].copy()

display(pd.DataFrame({
    "etapa": ["Datos originales", "Con precio no nulo", "Con precio <= 5000"],
    "filas": [len(df_raw), df["price_num"].notna().sum(), len(df_clean)]
}))

